In [ ]:
import numpy as np
from sklearn.datasets import fetch_openml
import pandas as pd

fashion_mnist = fetch_openml('Fashion-MNIST', version=1, as_frame=False, parser='auto')
X, Y = fashion_mnist.data, fashion_mnist.target

#normalize data and one hot encode y
X = X / 255.0
Y = Y.astype(int)
#y = np.eye(10)[y] #apparently pytorch does this for you?

#applying z-score normalization: mean and std are known for fashion mnist
mean = 0.2860
std = 0.3530
X = (X - mean) / std

training_loss_overall = []
validation_loss_overall = []
training_accuracy_overall = []
validation_accuracy_overall = []
final_test = []


In [ ]:

# seeds used: 1,2,3
np.random.seed(3)
indices = np.random.permutation(len(Y))

X = X[indices]
Y = Y[indices]

#split main df to make training, testing and validation sets
test_set = X[0:7000] #10% of df, 7k data points
valid_set = X[7000:14000] #10% of df, 7k data points
train_set = X[14000:] #80% of df, 56k data points

#split training again
unlabelled_train_set = train_set[0:50400] #90% unlabelled
labelled_train_set = train_set[50400:] #10% labelled

#split ys
y_test = Y[0:7000]
y_valid = Y[7000:14000]
y_train = Y[14000:]

y_unlabelled_train = y_train[0:50400]
y_labelled_train = y_train[50400:]


ModuleNotFoundError: No module named 'torch'

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, TensorDataset

#conv to torch and move to cuda
test = torch.tensor(test_set, dtype=torch.float32)#.to('cuda')
valid = torch.tensor(valid_set, dtype=torch.float32)#.to('cuda')
train = torch.tensor(train_set, dtype=torch.float32)#.to('cuda')

y_test = torch.tensor(y_test, dtype=torch.long)#.to('cuda')
y_valid = torch.tensor(y_valid, dtype=torch.long)#.to('cuda')
y_train = torch.tensor(y_train, dtype=torch.long)#.to('cuda')

#conv the labeled/unlabeled
labelled_train_set = torch.tensor(labelled_train_set, dtype=torch.float32)#.to('cuda')
unlabelled_train_set = torch.tensor(unlabelled_train_set, dtype=torch.float32)
y_unlabelled_train = torch.tensor(y_unlabelled_train, dtype=torch.long)#.to('cuda')
y_labelled_train = torch.tensor(y_labelled_train, dtype=torch.long)#.to('cuda')

#nn class
class NeuralNetwork(torch.nn.Module):
  def __init__(self):
    super().__init__()
    self.flatten = torch.nn.Flatten()
    self.linear_relu_stack = torch.nn.Sequential(
      torch.nn.Linear(28*28,512),
      torch.nn.ReLU(),
      torch.nn.Linear(512,512),
      torch.nn.ReLU(),
      torch.nn.Linear(512,256),
      torch.nn.ReLU(),
      torch.nn.Linear(256,10)
    )


  def forward(self,x):
    x = self.flatten(x)
    logits = self.linear_relu_stack(x)
    return logits

model = NeuralNetwork()#.to('cuda')

"""this function checks if the current layer is a linear layer, if it is,
the weights are changed to He init. m represents current layer"""
def weights_init_he(m):
    if isinstance(m, torch.nn.Linear):
        torch.nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')

def weights_init_random_normal(m):
    if isinstance(m, torch.nn.Linear):
        torch.nn.init.normal_(m.weight,mean=0,std=0.01)

def weights_init_random_uniform(m):
    if isinstance(m, torch.nn.Linear):
        torch.nn.init.uniform_(m.weight,mode = 'fan_in',nonlinearity='relu')

#.apply() recursively visits every layer and applies this function
model.apply(weights_init_he)

#criterion = torch.nn.MSELoss()
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05,momentum = 0.5) #m = 0.5,0.99; lr = 0.001, 0.1
#optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

#to make minibatch gradient descent,
dataset = TensorDataset(labelled_train_set, y_labelled_train) #if this crashes check if the params are pytorch tensors
batch_size = 100
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

#for overfitting analysis, collect std and mean
n = 100 #num epochs
loss_every_epoch = np.zeros(n)

#datacollection
training_loss = []
validation_loss = []
training_accuracy = []
validation_accuracy = []
epochs_til_convergence = 0

#do an initial training and validation epoch
with torch.no_grad():
  outputs = model(labelled_train_set) #forward pass?
  loss = criterion(outputs, y_labelled_train) #calculates loss
  pred_probab = torch.nn.Softmax(dim=1)(outputs)
  y_pred = pred_probab.argmax(1)
  pred_probab = torch.nn.Softmax(dim=1)(outputs)
  numpy_pred = y_pred.cpu().detach().numpy()
  y_train_numpy = y_labelled_train.cpu().detach().numpy()
  acc = numpy_pred == y_train_numpy #actually not the accuracy, just t/f vals if there was a match
  training_accuracy.append(acc.astype(int).sum()/len(acc))
  training_loss.append(loss.item())

  outputs = model(valid)
  loss = criterion(outputs, y_valid)
  pred_probab = torch.nn.Softmax(dim=1)(outputs)
  y_pred = pred_probab.argmax(1)
  #validation loss
  validation_loss.append(loss.item())
  #validation accuracy
  pred_probab = torch.nn.Softmax(dim=1)(outputs)
  y_pred = pred_probab.argmax(1)
  numpy_pred = y_pred.cpu().detach().numpy()
  y_valid_numpy = y_valid.cpu().detach().numpy()
  acc = numpy_pred == y_valid_numpy
  validation_accuracy.append(acc.astype(int).sum()/len(acc))
#train
for epoch in range(n):
  model.train() #this is a training mode. switch to model.eval() for testing.

  #some vars for minibatch data storage
  running_mean_loss = 0 #stores average loss over minibatches, so this is mean per epoch
  running_mean_acc = 0 #stores acc loss over minibatches, so this is acc per epoch
  count = 1 #counts minibatches

  #loads minibatch
  for x,y in dataloader: #if you leave out this loop this stays as full batch gd?
      #x, y = x.to('cuda'), y.to('cuda') #DONT FORGET TO UNCOMMENT

      outputs = model(x) #forward pass?
      loss = criterion(outputs, y) #calculates loss

      optimizer.zero_grad() #
      loss.backward() #calculates gradients
      optimizer.step() #updates the weights

      #calc training loss and acc (based on avg of minibatches)
      pred_probab = torch.nn.Softmax(dim=1)(outputs)
      y_pred = pred_probab.argmax(1)

      numpy_pred = y_pred.cpu().detach().numpy()
      y_train_numpy = y.cpu().detach().numpy()

      acc = numpy_pred == y_train_numpy #actually not the accuracy, just t/f vals if there was a match

      #store average loss and acc of minibatches
      running_mean_loss = running_mean_loss + (loss.item() - running_mean_loss) / count
      running_mean_acc = running_mean_acc + (acc.astype(int).sum()/len(acc) - running_mean_acc) / count
      count += 1

  #store the minibatch averages as per-epoch averages
  training_loss.append(running_mean_loss)
  training_accuracy.append(running_mean_acc)


  if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch + 1}, Loss: {loss.item():.4f}')

  #validation
  model.eval()
  with torch.no_grad(): #this disables gradient calculations
    outputs = model(valid)
    loss = criterion(outputs, y_valid)

    pred_probab = torch.nn.Softmax(dim=1)(outputs)
    y_pred = pred_probab.argmax(1)

  #validation loss
  validation_loss.append(loss.item())

  #validation accuracy
  pred_probab = torch.nn.Softmax(dim=1)(outputs)
  y_pred = pred_probab.argmax(1)

  numpy_pred = y_pred.cpu().detach().numpy()
  y_valid_numpy = y_valid.cpu().detach().numpy()

  acc = numpy_pred == y_valid_numpy
  validation_accuracy.append(acc.astype(int).sum()/len(acc))

  loss_every_epoch[epoch] = loss.item()

  #check overfitting
  loss_every_epoch[loss_every_epoch == 0.0] = np.nan
  mean = np.nanmean(loss_every_epoch)
  std = np.nanstd(loss_every_epoch)
  if loss.item() > mean + std:
    print('overfitting')
    epochs_til_convergence = epoch
    break

logits = model(train)
pred_probab = torch.nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

training_loss_overall.append(training_loss)
validation_loss_overall.append(validation_loss)
training_accuracy_overall.append(training_accuracy)
validation_accuracy_overall.append(validation_accuracy)

logits = model(test)
pred_probab = torch.nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

numpy_pred = y_pred.cpu().detach().numpy()
y_test_numpy = y_test.cpu().detach().numpy()

d = numpy_pred == y_test_numpy
d.astype(int).sum()/len(d)
final_test.append(d.astype(int).sum()/len(d))

In [ ]:
logits = model(test)
pred_probab = torch.nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

numpy_pred = y_pred.cpu().detach().numpy()
y_test_numpy = y_test.cpu().detach().numpy()

d = numpy_pred == y_test_numpy
d.astype(int).sum()/len(d)

NameError: name 'model' is not defined